In [ ]:
from pathlib import Path
import pandas as pd

# Merge

In [ ]:
# Import cleaned district level data
DATA_DIR = Path("data/intermediate")

df_trust = pd.read_csv(DATA_DIR / "anes_district_level.csv", low_memory=False)
df_house = pd.read_csv(DATA_DIR / "house_election_district_level.csv", low_memory=False)
df_pres = pd.read_csv(DATA_DIR / "presidential_election_district_level.csv", low_memory=False)
df_cbp = pd.read_csv(DATA_DIR / "cbp_district_level.csv", low_memory=False)

for df in [df_trust, df_house, df_pres, df_cbp]:
    df.drop(columns=[c for c in df.columns if c.startswith("Unnamed:")], inplace=True)

In [ ]:
# Clean column names
cols_to_suffix = ['total_votes', 'vote_share', 'dem_share', 'dem_win_margin']

df_house = df_house.rename(
    columns=lambda c: f"{c}_h" if c in cols_to_suffix else c
)

df_pres = df_pres.rename(
    columns=lambda c: f"{c}_p" if c in cols_to_suffix else c
)

In [ ]:
# Define merge keys
MERGE_KEYS = ["year", "state_code", "district_code"]

# Unify merge keys format
for df in [df_trust, df_house, df_pres, df_cbp]:
    df[MERGE_KEYS] = df[MERGE_KEYS].astype("string")

In [ ]:
# Drop duplicate state_district_code
for df in [df_house, df_pres, df_cbp]:
    df.drop(columns=["state_district_code"], errors="ignore", inplace=True)

In [ ]:
# Merge based on ANES trust data
df_merged = (
    df_trust
    .merge(df_house, on=MERGE_KEYS, how="left", validate="one_to_one")
    .merge(df_pres,  on=MERGE_KEYS, how="left", validate="one_to_one")
    .merge(df_cbp,   on=MERGE_KEYS, how="left", validate="one_to_many")
)

In [ ]:
# Save cleaned, finalized, distaict level data
df_merged.to_csv("data/final/merged_district_level.csv", index=False)